In [ ]:
import os
import re
import json
import time
import random
import threading
from datetime import datetime
from pathlib import Path
from threading import Semaphore
from typing import Any, Dict, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# =========================
# ✅ CONFIG
# =========================
BEDROCK_API_KEY = ""

BEDROCK_REGION = os.getenv("BEDROCK_REGION", "ap-south-1").strip()

BEDROCK_BASE_URL = os.getenv(
    "BEDROCK_BASE_URL",
    f"https://bedrock-mantle.{BEDROCK_REGION}.api.aws/v1",
).strip()

BEDROCK_MODEL = os.getenv("BEDROCK_MODEL", "google.gemma-3-27b-it").strip()

# Column that holds the property description text ("(ಎ) ಆಸ್ತಿ ವಿವರ ...").
# Leave as None to auto-detect (header containing "ಆಸ್ತಿ ವಿವರ"), or set an exact header name.
PROPERTY_COLUMN: Optional[str] = "(ಎ) ಆಸ್ತಿ ವಿವರ"
PROPERTY_COLUMN_KEYWORDS = ["ಆಸ್ತಿ ವಿವರ", "Property", "Schedule"]
FALLBACK_COLUMN_INDEX = 1  # 2nd column, as in the sample file

MAX_INPUT_CHARS = 6000  # guard against very long descriptions

client = OpenAI(
    api_key=BEDROCK_API_KEY,
    base_url=BEDROCK_BASE_URL,
    timeout=180.0,
    max_retries=0,
)


# =========================
# ✅ RATE LIMITER
# =========================
class AdvancedRateLimiter:
    """Sliding-window RPM limiter + parallel slot semaphore + token counters."""

    def __init__(self, max_requests: int = 28, period_seconds: int = 60, max_parallel: int = 10):
        self.max_requests = max_requests
        self.period_seconds = period_seconds
        self.max_parallel = min(max_parallel, max_requests)
        self.requests = []
        self.lock = threading.Lock()
        self.semaphore = Semaphore(self.max_parallel)

        self.total_requests = 0
        self.rate_limit_hits = 0

        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_consumed_tokens = 0

        print(f"⚡ Rate limiter configured: {self.max_parallel} parallel requests, {max_requests} RPM")

    def acquire_slot(self):
        self.semaphore.acquire()

    def release_slot(self):
        self.semaphore.release()

    def wait_if_needed(self):
        """Block until a request may be sent. Returns (timestamp, request_number)."""
        while True:
            with self.lock:
                now = time.time()
                self.requests = [t for t in self.requests if now - t < self.period_seconds]

                if len(self.requests) < self.max_requests:
                    self.requests.append(now)
                    self.total_requests += 1
                    return now, self.total_requests

                self.rate_limit_hits += 1
                wait_time = self.period_seconds - (now - self.requests[0]) + 0.05

            # sleep OUTSIDE the lock so other threads can still record tokens
            if wait_time > 0:
                print(f"⏰ Rate limit reached. Waiting {wait_time:.1f}s...")
                time.sleep(wait_time)

    def record_token_usage(self, input_tokens, output_tokens, total_tokens) -> Dict[str, int]:
        with self.lock:
            self.total_input_tokens += int(input_tokens or 0)
            self.total_output_tokens += int(output_tokens or 0)
            self.total_consumed_tokens += int(total_tokens or 0)
            return {
                "total_input_tokens": self.total_input_tokens,
                "total_output_tokens": self.total_output_tokens,
                "total_consumed_tokens": self.total_consumed_tokens,
            }

    def get_status(self):
        with self.lock:
            now = time.time()
            recent = [t for t in self.requests if now - t < self.period_seconds]
            return {
                "recent_requests": len(recent),
                "total_requests": self.total_requests,
                "rate_limit_hits": self.rate_limit_hits,
                "total_input_tokens": self.total_input_tokens,
                "total_output_tokens": self.total_output_tokens,
                "total_consumed_tokens": self.total_consumed_tokens,
            }


rate_limiter = AdvancedRateLimiter(max_requests=28, period_seconds=60, max_parallel=10)


# =========================
# ✅ OUTPUT SCHEMA
# =========================
EXPECTED_FIELDS = [
    # Translation of property details
    "property_details_en",
    # Location
    "hobli_en", "hobli_original",
    "village_en", "village_original",
    "taluka_en", "taluka_original",
    "district_en", "district_original",
    "grama_panchayat_en", "grama_panchayat_original",
    "ward_no", "locality_layout_en", "locality_layout_original",
    "full_address_en", "full_address_original",
    # Property identification
    "property_type", "property_sub_type", "land_use", "unit_configuration",
    "property_no", "site_no", "plot_no", "flat_no", "floor_no",
    "survey_no", "old_survey_no", "new_survey_no", "hissa_no",
    "khata_no", "e_khata_no", "pid_no", "katha_extract_no",
    "conversion_order_no", "conversion_order_date",
    "project_name_en", "project_name_original",
    # Dimensions & area (as written & calculated)
    "east_west_dimension", "north_south_dimension",
    "area_sqft", "area_sqm", "area_acres", "area_guntas", "area_as_written",
    "built_up_area_sqft", "built_up_area_sqm",
    "total_area_sqft_calc", "total_area_sqm_calc",
    "is_vacant", "building_description",
    "road_width",
    # Boundaries
    "boundary_east_en", "boundary_east_original",
    "boundary_west_en", "boundary_west_original",
    "boundary_north_en", "boundary_north_original",
    "boundary_south_en", "boundary_south_original",
    # Owner named in schedule (rare)
    "owner_name_in_schedule",
    "remarks",
]

# Fields computed in Python after the LLM call
COMPUTED_FIELDS = ["total_area_sqft_calc", "total_area_sqm_calc"]

NULL_LIKE = {"", "null", "none", "na", "n/a", "nm", "nil", "-", "*", ".", "not mentioned", "not available"}


def get_empty_structure() -> Dict[str, Any]:
    d = {k: None for k in EXPECTED_FIELDS}
    d.update({k: None for k in COMPUTED_FIELDS})
    return d


# =========================
# ✅ PREPROCESSING & POSTPROCESSING HELPERS
# =========================
def regex_prefill(text: str) -> Dict[str, Any]:
    """Deterministic fields that don't need an LLM. Used as fallback / cross-check."""
    out: Dict[str, Any] = {}

    m = re.search(r"Hobli:\s*([^,\[]*?)\s*,\s*Village:\s*([^,\[]*?)\s*,\s*Property number:\s*([^\s.]*)", text)
    if m:
        out["hobli_en"] = m.group(1).strip() or None
        out["village_en"] = m.group(2).strip() or None
        out["property_no"] = m.group(3).strip() or None

    # Karnataka PID / E-Katha numbers are 18 digits (some 15-20); pick the first long digit run
    m = re.search(r"(?<!\d)(\d{15,20})(?!\d)", text)
    if m:
        out["pid_no"] = m.group(1)

    m = re.search(r"Hobli Name:\s*([^,\[]+)", text)
    if m and not out.get("hobli_en"):
        out["hobli_en"] = m.group(1).strip()

    return out


def _to_float(v) -> Optional[float]:
    if v is None:
        return None
    try:
        return float(re.sub(r"[^\d.]", "", str(v)))
    except ValueError:
        return None


def compute_total_area(rec: Dict[str, Any]):
    """Convert whatever area was extracted into sq ft and sq m.
    1 sq m = 10.7639 sq ft | 1 acre = 43,560 sq ft | 1 gunta = 1,089 sq ft."""
    sqft = _to_float(rec.get("area_sqft"))
    sqm = _to_float(rec.get("area_sqm"))
    acres = _to_float(rec.get("area_acres")) or 0.0
    guntas = _to_float(rec.get("area_guntas")) or 0.0

    if sqft:
        total_sqft = sqft
    elif sqm:
        total_sqft = sqm * 10.7639
    elif acres or guntas:
        total_sqft = acres * 43560 + guntas * 1089
    else:
        total_sqft = None

    if total_sqft:
        rec["total_area_sqft_calc"] = round(total_sqft, 2)
        rec["total_area_sqm_calc"] = round(total_sqft / 10.7639, 2)


def clean_record(data: Dict[str, Any], text: str) -> Dict[str, Any]:
    cleaned = get_empty_structure()

    for key, value in data.items():
        if isinstance(value, (list, dict)):
            value = json.dumps(value, ensure_ascii=False)
        if isinstance(value, str):
            value = value.strip()
            if value.lower() in NULL_LIKE:
                value = None
        if key in cleaned or key.startswith("_"):
            cleaned[key] = value

    # fall back to regex for anything the LLM missed
    for k, v in regex_prefill(text).items():
        if not cleaned.get(k):
            cleaned[k] = v

    compute_total_area(cleaned)
    return cleaned


# =========================
# ✅ LLM EXTRACTION
# =========================
SYSTEM_PROMPT = (
    "You are a high-precision Karnataka land-records (IGR / Index-II) extraction engine. "
    "You read mixed Kannada + English property descriptions and return only valid JSON."
)


def build_prompt(text: str) -> str:
    return f"""
Extract structured details from the following Karnataka IGR "Property Description" (ಆಸ್ತಿ ವಿವರ) text.
The text is usually English with some Kannada mixed in. It may contain sections such as
"Hobli: ..., Village: ..., Property number: ...", "[BoundaryDetails]", "[Schedule details]",
"[LAND MARK]", "[EAST]/[WEST]/[NORTH]/[SOUTH]" and "Note: [Schedule A: ...]".

**CRITICAL RULES**
1. Return ONLY one valid JSON object with exactly the keys listed below. No explanations, no markdown.
2. Use null for anything not present. Placeholders such as "NM", "NA", "-", "*", "0", "ODD SITE" mean not measured -> null
   (for dimensions). Do NOT invent values.
3. For fields with _en and _original: `_original` = text exactly as written (Kannada script if written in Kannada,
   otherwise the English text as written); `_en` = English translation/transliteration
   (keep proper nouns as transliterations, e.g. "ಆನೇಕಲ್" -> "Anekal").
4. Keep all numbers exactly as written (survey numbers, khata numbers, PID, site numbers). Do not merge or reformat them.
5. Area: extract what is stated. area_sqft / area_sqm / area_acres / area_guntas must be plain numbers (no units).
   Use the TOTAL area of the property described (not a parent parcel from which it is carved out, unless that
   is the only figure). "1 Acre 06 Guntas" -> area_acres = 1, area_guntas = 6. Put the original wording in area_as_written.
   Do not convert between units yourself.
6. Multiple properties / items in one description: use the FIRST/main property for scalar fields and mention the
   others briefly in `remarks`.
7. property_details_en: Provide a complete and accurate English translation of the full property description text (translating Kannada terms into English standard real estate terms while transliterating proper names).
8. unit_configuration: Extract apartment/house BHK or unit configuration if mentioned (e.g. "1 BHK", "2 BHK", "3 BHK", "4 BHK", "1 RK", "Studio", "Duplex"). Normalize to standard format like "2 BHK". Use null if not mentioned or for vacant land/plots.

**KANNADA GLOSSARY**
ನಿವೇಶನ = Site/Plot | ಖಾಲಿ ನಿವೇಶನ = Vacant site | ಖಾತಾ = Khata | ಇ-ಖಾತಾ = E-Khata | ಸರ್ವೆ ನಂ = Survey No |
ಮಾಜಿ/ಹಳೇ = Old | ಹಾಲಿ/ಹೊಸ = New | ಎಕರೆ = Acre | ಗುಂಟೆ = Gunta | ಚದರ ಅಡಿ/ಚದರಡಿ = Sq ft | ಚದರ ಮೀಟರ್ = Sq m |
ಪೂರ್ವ = East | ಪಶ್ಚಿಮ = West | ಉತ್ತರ = North | ದಕ್ಷಿಣ = South | ರಸ್ತೆ = Road | ಖಾಸಗಿ ಸ್ವತ್ತು = Private property |
ಗ್ರಾಮ ಪಂಚಾಯಿತಿ = Gram Panchayat | ಹೋಬಳಿ = Hobli | ತಾಲ್ಲೂಕು = Taluk | ಜಿಲ್ಲೆ = District | ಬಡಾವಣೆ = Layout |
ಭೂಪರಿವರ್ತನೆ = Land conversion | ಕಟ್ಟಡ = Building | ಮನೆ = House | ಅಪಾರ್ಟ್‌ಮೆಂಟ್/ಫ್ಲಾಟ್ = Apartment/Flat

**FIELDS**
GENERAL:
- property_details_en: complete English translation of the entire property details text

LOCATION:
- hobli_en, hobli_original
- village_en, village_original
- taluka_en, taluka_original
- district_en, district_original
- grama_panchayat_en, grama_panchayat_original
- ward_no
- locality_layout_en, locality_layout_original: layout/colony/extension name (e.g. "BMRDA Layout", "Chepuru Layout")
- full_address_en: one-line English address built from the schedule
- full_address_original: schedule/address text as written

PROPERTY IDENTIFICATION:
- property_type: one of "Site", "Agricultural Land", "Converted Land", "House", "Flat/Apartment", "Building", "Commercial", "Other"
- property_sub_type: e.g. "Vacant site", "RCC building", "Dry land", "Layout site", "Undivided share"
- land_use: "Residential", "Commercial", "Agricultural", "Industrial", "Mixed", etc.
- unit_configuration: unit BHK/configuration if stated (e.g. "1 BHK", "2 BHK", "3 BHK", "4 BHK", "1 RK", "Studio", "Duplex", etc.; null if not mentioned)
- property_no: the "Property number" / Municipal/Panchayat property no.
- site_no, plot_no, flat_no, floor_no
- survey_no: as written (e.g. "63", "5/3/2/41"); old_survey_no, new_survey_no if both are stated; hissa_no
- khata_no (Khatha/Katha/ಖಾತಾ number, as written)
- e_khata_no: E-Katha number (long digit string)
- pid_no: Property Identification Number / PID (long digit string)
- katha_extract_no: Form-9/11 / Namune number if stated
- conversion_order_no, conversion_order_date
- project_name_en, project_name_original: apartment / layout / project name if any

DIMENSIONS & AREA:
- east_west_dimension, north_south_dimension: with unit, e.g. "40 feet"
- area_sqft, area_sqm, area_acres, area_guntas (numbers only), area_as_written
- built_up_area_sqft, built_up_area_sqm (numbers only, for buildings)
- is_vacant: true / false / null
- building_description: e.g. "RCC building, ground + first floor"
- road_width: e.g. "20 feet"

BOUNDARIES (from [BoundaryDetails], "east:/west:/north:/south:", or [EAST]/[WEST] blocks):
- boundary_east_en, boundary_east_original
- boundary_west_en, boundary_west_original
- boundary_north_en, boundary_north_original
- boundary_south_en, boundary_south_original

OTHER:
- owner_name_in_schedule: owner named inside the schedule text, if any
- remarks: anything important not captured above (multiple items, share fractions, conversion details, etc.)

**TEXT TO PARSE:**
\"\"\"{text}\"\"\"

Return ONLY the JSON object.
""".strip()


def _strip_fences(content: str) -> str:
    content = content.strip()
    if content.startswith("```"):
        lines = content.split("\n")
        if lines[-1].strip().startswith("```"):
            lines = lines[1:-1]
        else:
            lines = lines[1:]
        content = "\n".join(lines).strip()
    return content


def _salvage_json(content: str) -> Optional[dict]:
    start, end = content.find("{"), content.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        return json.loads(content[start:end + 1])
    except Exception:
        return None


def extract_all_details_bedrock(text: str, retries: int = 3) -> Dict[str, Any]:
    """Extract structured fields from one Karnataka IGR property description."""
    text = (text or "").strip()
    if not text:
        return get_empty_structure()
    text = text[:MAX_INPUT_CHARS]

    prompt = build_prompt(text)
    content = ""

    for attempt in range(1, retries + 1):
        rate_limiter.acquire_slot()
        try:
            request_time, request_num = rate_limiter.wait_if_needed()
            if request_num % 5 == 0 or attempt > 1:
                print(f"📤 Request #{request_num} at {datetime.fromtimestamp(request_time).strftime('%H:%M:%S')}")

            response = client.chat.completions.create(
                model=BEDROCK_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                temperature=0,
                max_tokens=4096,
            )
            content = (response.choices[0].message.content or "").strip()

            usage = response.usage
            prompt_tokens = usage.prompt_tokens if usage else 0
            completion_tokens = usage.completion_tokens if usage else 0
            total_tokens = usage.total_tokens if usage else 0

            cumulative = rate_limiter.record_token_usage(prompt_tokens, completion_tokens, total_tokens)
            tqdm.write(
                f"🪙 Req #{request_num} | In: {prompt_tokens:,} | Out: {completion_tokens:,} | "
                f"Total: {total_tokens:,} | Cumulative: {cumulative['total_consumed_tokens']:,}"
            )

            content = _strip_fences(content)
            try:
                data = json.loads(content)
            except json.JSONDecodeError:
                data = _salvage_json(content)
                if data is None:
                    raise

            cleaned = clean_record(data, text)
            cleaned["_token_usage"] = {
                "request_number": request_num,
                "input_tokens": prompt_tokens,
                "output_tokens": completion_tokens,
                "total_tokens": total_tokens,
            }
            return cleaned

        except json.JSONDecodeError as e:
            print(f"[Attempt {attempt}] JSON parse error: {e}")
            print(f"Raw response: {content[:300]}...")
        except Exception as e:
            print(f"[Attempt {attempt}] Error: {e}")
        finally:
            rate_limiter.release_slot()

        if attempt < retries:
            wait_time = min(60, 2 ** attempt + random.randint(1, 10))
            print(f"⏳ Waiting {wait_time}s before retry...")
            time.sleep(wait_time)

    # all retries failed -> empty structure, but keep regex-derived fields
    fallback = clean_record({}, text)
    fallback["_extraction_failed"] = True
    return fallback


# =========================
# ✅ COLUMN DETECTION
# =========================
def find_property_column(df: pd.DataFrame) -> str:
    if PROPERTY_COLUMN:
        if PROPERTY_COLUMN not in df.columns:
            raise ValueError(f"PROPERTY_COLUMN '{PROPERTY_COLUMN}' not found. Available: {list(df.columns)}")
        return PROPERTY_COLUMN

    for kw in PROPERTY_COLUMN_KEYWORDS:
        for col in df.columns:
            if kw in str(col):
                return col

    if len(df.columns) > FALLBACK_COLUMN_INDEX:
        col = df.columns[FALLBACK_COLUMN_INDEX]
        print(f"⚠️ Property column not matched by name; using column #{FALLBACK_COLUMN_INDEX + 1}: {col!r}")
        return col

    raise ValueError(f"Could not find property description column. Available: {list(df.columns)}")


# =========================
# ✅ PARALLEL PROCESSING
# =========================
def parallel_extract_bedrock(df_chunk: pd.DataFrame, text_col: str, max_workers: int = 10) -> pd.DataFrame:
    total_items = len(df_chunk)
    print(f"🚀 Processing {total_items} items with {max_workers} parallel workers")

    def process_row(row_idx, row):
        text = row.get(text_col, "")
        if not isinstance(text, str):
            text = "" if pd.isna(text) else str(text)
        return row_idx, extract_all_details_bedrock(text)

    results = [None] * total_items

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {}
        for idx, (_, row) in enumerate(df_chunk.iterrows()):
            futures[executor.submit(process_row, idx, row)] = idx

        with tqdm(total=total_items, desc="Processing") as pbar:
            for future in as_completed(futures):
                row_idx = futures[future]
                try:
                    _, result = future.result(timeout=300)
                    results[row_idx] = result
                except Exception as e:
                    print(f"\n[Error] row {row_idx}: {e}")
                    results[row_idx] = get_empty_structure()

                status = rate_limiter.get_status()
                pbar.set_postfix({
                    "RPM": f"{status['recent_requests']}/{rate_limiter.max_requests}",
                    "Requests": status["total_requests"],
                    "Tokens": f"{status['total_consumed_tokens']:,}",
                })
                pbar.update(1)

    # Flatten: one column per extracted field + token columns
    base = df_chunk.reset_index(drop=True).copy()
    flat_rows = []
    for r in results:
        r = dict(r or get_empty_structure())
        usage = r.pop("_token_usage", {}) or {}
        r["request_number"] = usage.get("request_number")
        r["input_tokens"] = usage.get("input_tokens", 0)
        r["output_tokens"] = usage.get("output_tokens", 0)
        r["total_tokens"] = usage.get("total_tokens", 0)
        r["extraction_failed"] = bool(r.pop("_extraction_failed", False))
        flat_rows.append(r)

    flat_df = pd.DataFrame(flat_rows)
    ordered = EXPECTED_FIELDS + [
        "request_number", "input_tokens", "output_tokens", "total_tokens", "extraction_failed"
    ]
    seen = set()
    ordered_unique = [x for x in ordered if not (x in seen or seen.add(x))]
    flat_df = flat_df.reindex(columns=ordered_unique)

    return pd.concat([base, flat_df], axis=1)


# =========================
# ✅ TOKEN LEDGER (one workbook per chunk)
# =========================
def save_chunk_token_ledger(result_df: pd.DataFrame, chunk_idx: int, output_dir: str,
                            source_start_row: int = 0) -> Path:
    n = len(result_df)
    details_df = pd.DataFrame({
        "chunk_number": chunk_idx,
        "row_in_chunk": range(1, n + 1),
        "source_excel_row": [source_start_row + i + 2 for i in range(n)],
        "request_number": result_df.get("request_number"),
        "input_tokens": pd.to_numeric(result_df.get("input_tokens"), errors="coerce").fillna(0).astype(int),
        "output_tokens": pd.to_numeric(result_df.get("output_tokens"), errors="coerce").fillna(0).astype(int),
        "total_tokens": pd.to_numeric(result_df.get("total_tokens"), errors="coerce").fillna(0).astype(int),
    })
    details_df["status"] = details_df["total_tokens"].apply(lambda t: "Success" if t > 0 else "No token data")

    ok = int((details_df["total_tokens"] > 0).sum())
    tot_in = int(details_df["input_tokens"].sum())
    tot_out = int(details_df["output_tokens"].sum())
    tot = int(details_df["total_tokens"].sum())

    summary_df = pd.DataFrame([{
        "chunk_number": chunk_idx,
        "records_in_chunk": n,
        "successful_records": ok,
        "records_without_token_data": n - ok,
        "total_input_tokens": tot_in,
        "total_output_tokens": tot_out,
        "total_tokens": tot,
        "average_input_tokens_per_record": round(tot_in / ok, 2) if ok else 0,
        "average_output_tokens_per_record": round(tot_out / ok, 2) if ok else 0,
        "average_total_tokens_per_record": round(tot / ok, 2) if ok else 0,
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])

    ledger_path = Path(output_dir) / f"token_ledger_chunk_{chunk_idx}.xlsx"
    with pd.ExcelWriter(ledger_path, engine="openpyxl") as writer:
        details_df.to_excel(writer, sheet_name="Token_Details", index=False)
        summary_df.to_excel(writer, sheet_name="Chunk_Summary", index=False)

    print(f"🧾 Token ledger saved: {ledger_path} | In: {tot_in:,} | Out: {tot_out:,} | Total: {tot:,}")
    return ledger_path


def merge_token_ledgers(output_dir: str, final_ledger_path: str = "token_ledger_all_chunks.xlsx") -> pd.DataFrame:
    ledger_files = sorted(
        Path(output_dir).glob("token_ledger_chunk_*.xlsx"),
        key=lambda p: int(re.search(r"token_ledger_chunk_(\d+)\.xlsx", p.name).group(1)),
    )
    if not ledger_files:
        print(f"⚠️ No token ledger files found in {output_dir}")
        return pd.DataFrame()

    all_details = pd.concat(
        [pd.read_excel(f, sheet_name="Token_Details") for f in ledger_files], ignore_index=True
    )

    per_chunk = all_details.groupby("chunk_number", as_index=False).agg(
        records_in_chunk=("row_in_chunk", "count"),
        successful_records=("total_tokens", lambda s: int((s > 0).sum())),
        total_input_tokens=("input_tokens", "sum"),
        total_output_tokens=("output_tokens", "sum"),
        total_tokens=("total_tokens", "sum"),
    )
    per_chunk["average_total_tokens_per_record"] = (
        per_chunk["total_tokens"] / per_chunk["successful_records"].replace(0, pd.NA)
    ).fillna(0).round(2)

    success_mask = all_details["total_tokens"] > 0
    grand = pd.DataFrame([{
        "total_chunks": len(per_chunk),
        "total_records": len(all_details),
        "successful_records": int(success_mask.sum()),
        "total_input_tokens": int(all_details["input_tokens"].sum()),
        "total_output_tokens": int(all_details["output_tokens"].sum()),
        "total_tokens": int(all_details["total_tokens"].sum()),
        "average_total_tokens_per_successful_record": round(
            all_details.loc[success_mask, "total_tokens"].mean(), 2
        ) if success_mask.any() else 0,
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])

    with pd.ExcelWriter(final_ledger_path, engine="openpyxl") as writer:
        all_details.to_excel(writer, sheet_name="All_Token_Details", index=False)
        per_chunk.to_excel(writer, sheet_name="Per_Chunk_Summary", index=False)
        grand.to_excel(writer, sheet_name="Grand_Summary", index=False)

    print(f"✅ Combined token ledger saved: {final_ledger_path}")
    return all_details


# =========================
# ✅ CHUNK PROCESSOR
# =========================
def process_chunk(chunk_df: pd.DataFrame, text_col: str, chunk_idx: int, total_chunks: int) -> pd.DataFrame:
    chunk_size = len(chunk_df)
    max_workers = min(rate_limiter.max_parallel, chunk_size) if chunk_size > 0 else 1

    print(f"\n{'=' * 60}\n📦 Processing Chunk {chunk_idx}/{total_chunks}\n{'=' * 60}")
    print(f"📊 Chunk size: {chunk_size} records | ⚡ Workers: {max_workers}")

    start = time.time()
    result_df = parallel_extract_bedrock(chunk_df, text_col, max_workers=max_workers)
    elapsed = time.time() - start

    rpm = (chunk_size / elapsed) * 60 if elapsed > 0 else 0
    print(f"\n✅ Chunk {chunk_idx} done in {elapsed:.1f}s ({rpm:.1f} RPM, target ≤{rate_limiter.max_requests})")
    print(f"   Rate limit hits: {rate_limiter.rate_limit_hits} | "
          f"Total tokens so far: {rate_limiter.total_consumed_tokens:,}")
    return result_df


# =========================
# ✅ MAIN PIPELINE
# =========================
def run_extraction_pipeline(input_path: str, output_dir: str = "output",
                            chunk_size: int = 60, resume: bool = True):
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    try:
        df = pd.read_excel(input_path) if input_path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(input_path)
    except Exception as e:
        raise ValueError(f"Error loading input file: {e}")

    text_col = find_property_column(df)
    print(f"📝 Using property description column: {text_col!r}")

    existing_chunks = set()
    if resume:
        for f in Path(output_dir).glob("extracted_chunk_*.xlsx"):
            m = re.search(r"extracted_chunk_(\d+)\.xlsx", f.name)
            if m:
                existing_chunks.add(int(m.group(1)))

    total_rows = len(df)
    total_chunks = (total_rows + chunk_size - 1) // chunk_size
    est_minutes = (total_rows + rate_limiter.max_requests - 1) // rate_limiter.max_requests

    print(f"\n{'=' * 60}\n🚀 KARNATAKA IGR EXTRACTION JOB\n{'=' * 60}")
    print(f"🤖 Model: {BEDROCK_MODEL}")
    print(f"📊 Records: {total_rows:,} | 📦 Chunk size: {chunk_size} | 📁 Chunks: {total_chunks}")
    print(f"⏱️ Rate limit: {rate_limiter.max_requests} RPM | ⚡ Parallel: {rate_limiter.max_parallel}")
    print(f"🕐 Estimated time: ~{est_minutes} min ({est_minutes / 60:.1f} h)\n{'=' * 60}\n")

    for chunk_idx in range(1, total_chunks + 1):
        output_file = Path(output_dir) / f"extracted_chunk_{chunk_idx}.xlsx"

        if resume and chunk_idx in existing_chunks:
            print(f"✓ Chunk {chunk_idx}/{total_chunks} already processed (skipping)")
            continue

        start_idx = (chunk_idx - 1) * chunk_size
        end_idx = min(chunk_idx * chunk_size, total_rows)
        chunk_df = df.iloc[start_idx:end_idx].copy()

        try:
            print(f"\n📂 Loading chunk {chunk_idx} (rows {start_idx:,}-{end_idx - 1:,})...")
            result_df = process_chunk(chunk_df, text_col, chunk_idx, total_chunks)
            result_df.to_excel(output_file, index=False)
            print(f"💾 Saved chunk {chunk_idx} to {output_file}")
            save_chunk_token_ledger(result_df, chunk_idx, output_dir, source_start_row=start_idx)

        except Exception as e:
            print(f"✗ Error processing chunk {chunk_idx}: {e}")
            empty = pd.DataFrame([get_empty_structure() for _ in range(len(chunk_df))])
            pd.concat([chunk_df.reset_index(drop=True), empty], axis=1).to_excel(output_file, index=False)
            print(f"⚠️ Saved empty results for chunk {chunk_idx}")

        if chunk_idx < total_chunks:
            time.sleep(2)

    print(f"\n{'=' * 60}\n🎉 EXTRACTION COMPLETE!\n{'=' * 60}")
    print(f"   Requests: {rate_limiter.total_requests:,} | Rate limit hits: {rate_limiter.rate_limit_hits}")
    print(f"   Input tokens: {rate_limiter.total_input_tokens:,}")
    print(f"   Output tokens: {rate_limiter.total_output_tokens:,}")
    print(f"   Total tokens: {rate_limiter.total_consumed_tokens:,}")


# =========================
# ✅ MERGE & ANALYZE
# =========================
def merge_results(output_dir: str, final_output_path: str = "final_output.xlsx") -> pd.DataFrame:
    chunk_files = sorted(
        Path(output_dir).glob("extracted_chunk_*.xlsx"),
        key=lambda x: int(re.search(r"extracted_chunk_(\d+)\.xlsx", x.name).group(1)),
    )
    if not chunk_files:
        raise ValueError(f"No chunk files found in {output_dir}")

    print(f"\n📂 Merging {len(chunk_files)} chunk files...")
    final_df = pd.concat([pd.read_excel(f) for f in tqdm(chunk_files, desc="Loading chunks")],
                         ignore_index=True)
    final_df.to_excel(final_output_path, index=False)
    print(f"✅ Merged into {final_output_path} | Total records: {len(final_df):,}")
    return final_df


def analyze_results(df: pd.DataFrame):
    print("\n" + "=" * 60 + "\n📈 EXTRACTION ANALYSIS\n" + "=" * 60)

    key_fields = [
        "property_details_en", "village_en", "property_type", "unit_configuration", "site_no", "survey_no", "khata_no",
        "pid_no", "area_sqft", "total_area_sqft_calc",
        "boundary_east_en", "boundary_north_en", "full_address_en",
    ]
    for field in key_fields:
        if field in df.columns:
            non_null = df[field].notna().sum()
            print(f"{field:25s}: {non_null:6d} / {len(df):6d} ({non_null / len(df) * 100:6.1f}%)")

    if "extraction_failed" in df.columns:
        print(f"\n⚠️ Failed extractions: {int(df['extraction_failed'].fillna(False).sum())}")

    if "property_type" in df.columns:
        print("\n🏠 PROPERTY TYPE DISTRIBUTION")
        for pt, cnt in df["property_type"].value_counts(dropna=False).head(10).items():
            print(f"{('Unknown' if pd.isna(pt) else pt):25s}: {cnt:6d} ({cnt / len(df) * 100:6.1f}%)")


# =========================
# ✅ MAIN
# =========================
if __name__ == "__main__":
    INPUT_FILE = r"E:\IGR\Karnataka_IGR\Processing\final_merged_scraped_data_raw.xlsx"
    OUTPUT_DIR = "extracted_results_karnataka"
    FINAL_OUTPUT = r"E:\IGR\Karnataka_IGR\Processing\final_merged_scraped_data_raw_llm_processed.xlsx"

    if not BEDROCK_API_KEY:
        raise SystemExit("❌ BEDROCK_API_KEY is missing. Paste your key at the top of this file.")

    try:
        run_extraction_pipeline(INPUT_FILE, output_dir=OUTPUT_DIR, chunk_size=60, resume=True)

        print("\n📦 Merging results...")
        final_df = merge_results(OUTPUT_DIR, FINAL_OUTPUT)

        print("\n🧾 Merging token ledgers...")
        merge_token_ledgers(OUTPUT_DIR, final_ledger_path="token_ledger_all_chunks.xlsx")

        analyze_results(final_df)
        print(f"\n✅ Results saved to: {FINAL_OUTPUT}")

    except KeyboardInterrupt:
        print(f"\n⚠️ Interrupted. Partial results are in {OUTPUT_DIR}/ — re-run to resume.")
    except Exception as e:
        import traceback
        print(f"\n❌ Error: {e}")
        traceback.print_exc()

⚡ Rate limiter configured: 10 parallel requests, 28 RPM
📝 Using property description column: '(ಎ) ಆಸ್ತಿ ವಿವರ'

🚀 KARNATAKA IGR EXTRACTION JOB
🤖 Model: google.gemma-3-27b-it
📊 Records: 14,708 | 📦 Chunk size: 60 | 📁 Chunks: 246
⏱️ Rate limit: 28 RPM | ⚡ Parallel: 10
🕐 Estimated time: ~526 min (8.8 h)


📂 Loading chunk 1 (rows 0-59)...

📦 Processing Chunk 1/246
📊 Chunk size: 60 records | ⚡ Workers: 10
🚀 Processing 60 items with 10 parallel workers
📤 Request #5 at 17:44:34
📤 Request #10 at 17:44:34


Processing:   2%|▏         | 1/60 [00:13<13:02, 13.26s/it, RPM=11/28, Requests=11, Tokens=2,672]

🪙 Req #6 | In: 1,745 | Out: 927 | Total: 2,672 | Cumulative: 2,672


Processing:   3%|▎         | 2/60 [00:13<05:41,  5.89s/it, RPM=12/28, Requests=12, Tokens=5,279]

🪙 Req #9 | In: 1,734 | Out: 873 | Total: 2,607 | Cumulative: 5,279


...

🪙 Req #238 | In: 1,897 | Out: 1,129 | Total: 3,026 | Cumulative: 686,466
📤 Request #240 at 17:53:30


Processing:  93%|█████████▎| 56/60 [02:01<00:12,  3.12s/it, RPM=23/28, Requests=242, Tokens=689,680]

🪙 Req #237 | In: 2,007 | Out: 1,207 | Total: 3,214 | Cumulative: 689,680


Processing:  93%|█████████▎| 56/60 [02:12<00:09,  2.36s/it, RPM=23/28, Requests=242, Tokens=689,680]


🪙 Req #239 | In: 1,922 | Out: 1,114 | Total: 3,036 | Cumulative: 692,716
🪙 Req #242 | In: 2,149 | Out: 1,047 | Total: 3,196 | Cumulative: 695,912
🪙 Req #240 | In: 1,996 | Out: 1,221 | Total: 3,217 | Cumulative: 699,129
🪙 Req #241 | In: 1,962 | Out: 1,167 | Total: 3,129 | Cumulative: 702,258

⚠️ Interrupted. Partial results are in extracted_results_karnataka/ — re-run to resume.
